# Track 3 · 01 Sleep-EDF prepare — the warm-up data onto Drive, as the harness makes it

**One job.** Put the warm-up corpus on Drive in the form `track3_02_eegnet` trains on:
the harness's own 5-second windows, their targets, and its train/val/test split.

Self-contained: upload this one file.

---

## The task, from source

The warm-up scores on **Sleep-EDF sleep-cassette** (neuralfetch `Kemp2000Analysis`): 78
people, 153 nights, 2 EEG channels (Fpz-Cz, Pz-Oz) at 100 Hz. Each window is 5 s and the
target is **seconds until the first N2 sleep epoch**, capped at 600:

```
target = clip(n2_onset - window_stop, 0, 600)
```

Windows tile the 20 minutes before N2 onset, every 5 s, so a typical night gives 240 of
them. That rule and the split are read from `neuralbench/tasks/eeg/sleep_onset/config.yaml`
and `AddSleepOnsetTargets` in `neuralbench/transforms.py`.

## Preprocessing is the harness's, exactly

Same reason as Track 1: the scoring harness prepares data with neuralbench's default
pipeline, so a model is only scored fairly on data prepared the same way.

| step | value | at 100 Hz |
|---|---|---|
| notch | 50 and 60 Hz plus harmonics below Nyquist | nothing qualifies, skipped |
| band-pass | 0.1 – 75 Hz | 75 Hz is above Nyquist, so only the 0.1 Hz highpass runs |
| resample | 120 Hz | 600 samples per window |
| scale | sklearn `RobustScaler` per channel over the whole recording | |
| baseline | none for this task | |
| clamp | ±20 | |

This code was checked against neuralset's own extractor on three nights: identical
windows (max difference 0.0) and identical targets.

## The split

The harness splits people, not nights: `SklearnSplit` by subject, seed 33, 20% test and
20% validation, giving **46 train / 16 val / 16 test** people. The 16 test people are the
warm-up test set, so `track3_02` must never train on them. The split is recomputed here
and written into the manifest.

## Licence

Sleep-EDF is on PhysioNet under **ODC-By 1.0** (attribution required, commercial use
allowed), per the licence field in neuralfetch's study file. Cite Kemp et al. 2000 and
PhysioNet.

## Cost

About 7.6 GB downloaded to Colab's temporary disk, one night at a time, deleted after
use. Roughly 180 MB written to Drive. About 15–25 minutes, mostly download. It resumes:
nights already on Drive are skipped on a rerun.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'
WORKERS = 6          # nights downloaded in parallel
PROCESS_AT_ONCE = 2  # nights preprocessed at once; each needs ~2 GB of RAM

# The harness's settings. Do not tune these; matching them is the point.
SFREQ = 120.0
NOTCH = (50.0, 60.0)
BANDPASS = (0.1, 75.0)
CLAMP = 20.0
DURATION = 5.0       # window length, s
STRIDE = 5.0         # window step, s
MAX_PRE_N2_S = 1200.0
CAP_S = 600.0
BIN_EDGES = (0.0, 40.0, 90.0, 300.0, 600.0)

# Kemp2000Analysis.SUBJECTS: 0..82 minus five people the study leaves out.
SUBJECTS = [x for x in range(83) if x not in [39, 68, 69, 78, 79]]
SPLIT_SEED = 33

print('NOTEBOOK VERSION: track3_01 v2')

In [ ]:
%pip install -q mne

import importlib
for mod in ('mne', 'sklearn', 'numpy'):
    print(f'{mod:8} {importlib.import_module(mod).__version__}')

In [ ]:
import hashlib, json, math, re, shutil, threading, time, urllib.request, warnings
warnings.filterwarnings('ignore')
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np, mne
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
mne.set_log_level('ERROR')

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

OUT = Path(ROOT) / 'Track3' / 'sleep_edf'
WIN = OUT / 'windows'
WIN.mkdir(parents=True, exist_ok=True)
TMP = Path('/content/sleep_edf_tmp') if Path('/content').exists() else OUT.parent / '_tmp'
TMP.mkdir(parents=True, exist_ok=True)

# PhysioNet's public S3 mirror: the same files as physionet.org, much faster.
MIRROR = 'https://physionet-open.s3.amazonaws.com/sleep-edfx/1.0.0'
sums = urllib.request.urlopen(f'{MIRROR}/SHA256SUMS.txt', timeout=60).read().decode()
sha = {p.split('/')[-1]: h for h, p in (l.split() for l in sums.splitlines() if l.strip())
       if p.startswith('sleep-cassette/')}

# One PSG and one hypnogram per (person, night), matched the way the study's
# iter_timelines globs them: SC4<pp><n>*-PSG.edf and SC4<pp><n>*-Hypnogram.edf.
nights = []
for s in SUBJECTS:
    for n in (1, 2):
        psg = [f for f in sha if re.fullmatch(f'SC4{s:02d}{n}..-PSG\\.edf', f)]
        hyp = [f for f in sha if re.fullmatch(f'SC4{s:02d}{n}..-Hypnogram\\.edf', f)]
        if len(psg) == 1 and len(hyp) == 1:
            nights.append(dict(subject=s, night=n, psg=psg[0], hyp=hyp[0]))
        elif psg or hyp:
            raise RuntimeError(f'unexpected files for subject {s} night {n}: {psg} {hyp}')
print(f'{len(nights)} nights from {len({x["subject"] for x in nights})} people '
      f'(the study expects 153 from 78)')

## Step 1 — download, preprocess and window each night

Each night is downloaded, checked against PhysioNet's SHA-256, turned into windows, saved
to Drive, and deleted from temporary disk. Downloads run six at a time, but only two
nights are preprocessed at once: a whole ~23-hour recording peaks at about 2 GB of RAM,
and six together exhausted a standard Colab session. The function follows neuralfetch (channel
names and types), neuralbench (onset marker and target) and neuralset (filtering,
resampling, scaling, window tiling, clamp) in that code's order.

In [ ]:
def harness_windows(psg_path, hyp_path):
    """One night -> (X, y, starts, info), following neuralfetch, neuralbench and
    neuralset in their order. Returns X = None when the night yields no windows."""
    # First scored N2 epoch: AddSleepOnsetTargets on Kemp2000Analysis's hypnogram events.
    ann = mne.read_annotations(hyp_path)
    n2 = [float(o) for o, d in zip(ann.onset, ann.description) if d == 'Sleep stage 2']
    raw = mne.io.read_raw_edf(psg_path, preload=False)
    rec_stop = raw.n_times / raw.info['sfreq']
    info = dict(native_sfreq=float(raw.info['sfreq']), rec_s=rec_stop,
                n2_onset=min(n2) if n2 else None)
    if not n2:
        return None, None, None, {**info, 'skipped': 'no N2 epoch'}
    t_onset = min(n2)
    t0 = max(0.0, t_onset - MAX_PRE_N2_S)
    t_stop = min(t_onset, rec_stop)
    span = (t_stop - DURATION) - t0
    if t_stop <= t0 or span < -1e-10:
        return None, None, None, {**info, 'skipped': 'pre-N2 region shorter than a window'}
    n = math.floor(span / STRIDE + 1e-9) + 1
    starts = t0 + np.arange(n) * STRIDE

    # "EEG Fpz-Cz" -> type eeg, name Fpz-Cz; the harness picks type eeg.
    eeg = [c for c in raw.ch_names if c.split(' ')[0].lower() == 'eeg']
    raw.pick(eeg).load_data()
    raw.rename_channels({c: c.split(' ', 1)[1] for c in eeg})

    nyq = raw.info['sfreq'] / 2
    freqs = [f for base in NOTCH for f in np.arange(base, min(nyq, 301), base)]
    if freqs:                      # at 100 Hz there are none
        raw.notch_filter(freqs, phase='zero', filter_length='auto', verbose=False)
    lo, hi = BANDPASS
    raw.filter(lo, hi if hi < nyq else None, verbose=False)   # lowpass dropped at/above Nyquist
    raw.resample(SFREQ, verbose=False)
    d = RobustScaler().fit_transform(raw.get_data().T).T     # per channel, whole recording

    n_times = int(round(DURATION * SFREQ))
    idx = np.round(starts * SFREQ).astype(int)
    X = np.stack([d[:, i:i + n_times] for i in idx]).astype('float32')
    X = np.clip(X, -CLAMP, CLAMP)
    y = np.clip(t_onset - (starts + DURATION), 0.0, CAP_S).astype('float32')
    return X, y, starts, {**info, 'channels': list(raw.ch_names), 'n_windows': int(n)}


def fetch(name):
    path = TMP / name
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f'{MIRROR}/sleep-cassette/{name}', timeout=120) as r, \
                 open(path, 'wb') as f:
                shutil.copyfileobj(r, f, 1 << 20)
            if hashlib.sha256(path.read_bytes()).hexdigest() == sha[name]:
                return path
            raise IOError('checksum mismatch')
        except Exception as e:
            if attempt == 3:
                raise RuntimeError(f'{name}: {e}')
            time.sleep(5 * (attempt + 1))


BUSY = threading.Semaphore(PROCESS_AT_ONCE)


def do_night(x):
    rec = x['psg'][:6]                            # SC4<person><night>, e.g. SC4001
    dest = WIN / f'{rec}.npz'
    if dest.exists():
        return rec, 'kept', json.loads(str(np.load(dest)['info']))
    psg, hyp = fetch(x['psg']), fetch(x['hyp'])
    try:
        with BUSY:                                # downloads overlap, preprocessing does not
            X, y, starts, info = harness_windows(psg, hyp)
    finally:
        psg.unlink(missing_ok=True); hyp.unlink(missing_ok=True)
    info.update(subject=x['subject'], night=x['night'], psg=x['psg'], hyp=x['hyp'])
    if X is None:
        np.savez(dest, info=json.dumps(info))     # recorded, so a rerun skips it too
        return rec, 'no windows', info
    np.savez(dest, X=X, y=y, starts=starts, info=json.dumps(info))
    return rec, 'new', info


t0 = time.time()
done, failed = {}, {}
with ThreadPoolExecutor(WORKERS) as pool:
    futs = {pool.submit(do_night, x): x for x in nights}
    for i, f in enumerate(as_completed(futs), 1):
        x = futs[f]
        try:
            rec, status, info = f.result()
            done[rec] = info
            if i % 10 == 0 or i == len(nights):
                print(f'  {i:3d}/{len(nights)}  {rec} {status}  {time.time() - t0:.0f}s')
        except Exception as e:
            failed[x['psg']] = repr(e)
            print(f'  FAILED {x["psg"]}: {e}')
print(f'\n{len(done)} nights processed, {len(failed)} failed, {time.time() - t0:.0f}s')
if failed:
    print('Rerun this cell: finished nights are kept and only the failures are retried.')

## Step 2 — the harness's split, the manifest and the checks

`SklearnSplit` takes the people in the order they first appear in the study's events,
which is ascending subject ID, and calls `train_test_split` twice with seed 33. The
recomputed split is written into the manifest so `track3_02` uses exactly these people.

In [ ]:
subjects = sorted(SUBJECTS)                       # "Kemp2000Analysis/00".. sorts the same way
idx = np.arange(len(subjects))
trval, test = train_test_split(idx, test_size=0.2, random_state=SPLIT_SEED, shuffle=True)
train, val = train_test_split(trval, test_size=0.2 / 0.8, random_state=SPLIT_SEED, shuffle=True)
split = {**{subjects[i]: 'train' for i in train}, **{subjects[i]: 'val' for i in val},
         **{subjects[i]: 'test' for i in test}}
print({k: sum(v == k for v in split.values()) for k in ('train', 'val', 'test')}, 'people')
print('test (= warm-up test set):', sorted(s for s, v in split.items() if v == 'test'))


def bin_counts(y):
    b = np.digitize(y, BIN_EDGES[1:-1])
    return [int((b == k).sum()) for k in range(len(BIN_EDGES) - 1)]


recs, per_split, ys = {}, {}, {'train': [], 'val': [], 'test': []}
problems = []
for f in sorted(WIN.glob('SC4*.npz')):
    z = np.load(f)
    info = json.loads(str(z['info']))
    sp = split[info['subject']]
    entry = dict(info, split=sp, file=f.name)
    if 'X' in z:
        X, y, st = z['X'], z['y'], z['starts']
        entry['n_windows'] = len(y)
        ys[sp].append(y)
        if X.shape[1:] != (2, int(DURATION * SFREQ)):
            problems.append(f'{f.name}: shape {X.shape}')
        if not np.allclose(np.diff(st), STRIDE):
            problems.append(f'{f.name}: window starts not {STRIDE} s apart')
        if not (np.isfinite(X).all() and np.abs(X).max() <= CLAMP):
            problems.append(f'{f.name}: values out of range')
        if y[-1] != 0.0 and info['n2_onset'] <= info['rec_s']:
            problems.append(f'{f.name}: last window target {y[-1]}, expected 0')
    recs[f.stem] = entry

for sp in ('train', 'val', 'test'):
    y = np.concatenate(ys[sp]) if ys[sp] else np.zeros(0)
    per_split[sp] = dict(people=sum(v == sp for v in split.values()),
                         nights=sum(1 for r in recs.values() if r['split'] == sp and r.get('n_windows')),
                         windows=int(len(y)), per_bin=bin_counts(y))
    print(f'  {sp:5}  {per_split[sp]["people"]:2d} people  {per_split[sp]["nights"]:3d} nights  '
          f'{len(y):6,d} windows   per bin {per_split[sp]["per_bin"]}')

skipped = {k: r['skipped'] for k, r in recs.items() if 'skipped' in r}
print(f'\n{len(recs)} nights on Drive, {len(skipped)} without windows: {skipped or "none"}')
print(f'channels {sorted({tuple(r["channels"]) for r in recs.values() if "channels" in r})}')
print('checks:', 'all passed' if not problems else problems[:10])

(OUT / 'manifest.json').write_text(json.dumps(dict(
    source='Sleep-EDF sleep-cassette (PhysioNet sleep-edfx 1.0.0, neuralfetch Kemp2000Analysis)',
    licence='ODC-By-1.0', mirror=MIRROR,
    preprocessing=dict(sfreq=SFREQ, notch=NOTCH, bandpass=BANDPASS, scaler='RobustScaler',
                       baseline=None, clamp=CLAMP),
    windows=dict(duration=DURATION, stride=STRIDE, max_pre_n2_s=MAX_PRE_N2_S, cap_s=CAP_S,
                 target='clip(n2_onset - window_stop, 0, cap_s)'),
    bin_edges=BIN_EDGES,
    split=dict(seed=SPLIT_SEED, by='subject', subjects={str(k): v for k, v in split.items()}),
    per_split=per_split, recordings=recs, failed=failed), indent=1))
size = sum(f.stat().st_size for f in WIN.glob('*.npz')) / 1e6
print(f'\nmanifest -> {OUT / "manifest.json"}   windows on Drive: {size:.0f} MB')
expected = len(nights) == 153 and not failed and not problems
print('READY for track3_02' if expected else 'NOT READY: see the counts and checks above')